# 05 - Google Colab Trimodal Training

This notebook is designed to run on **Google Colab** with a high-end GPU (T4/V100/A100).

**Workflow**:
1. Mount Google Drive
2. Install dependencies
3. Upload / clone codebase
4. Organize patient data from Drive
5. Run staged trimodal training
6. Save checkpoints back to Drive

**Recommended runtime**: GPU (T4 or higher)

In [ ]:
# ====================================================================
# CELL 1: Mount Google Drive
# ====================================================================
from google.colab import drive
drive.mount('/content/drive')

# Set paths
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/DepressoSpeech"  # UPDATE THIS
LOCAL_PROJECT_DIR = "/content/DepressoSpeech"

import os
print(f"Drive path: {DRIVE_PROJECT_DIR}")
print(f"Local path: {LOCAL_PROJECT_DIR}")
print(f"Drive exists: {os.path.exists(DRIVE_PROJECT_DIR)}")

In [ ]:
# ====================================================================
# CELL 2: Install Dependencies
# ====================================================================
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q pandas numpy scipy scikit-learn librosa soundfile sentence-transformers transformers
!pip install -q pyyaml tqdm matplotlib

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ====================================================================
# CELL 3: Copy Project from Drive (or clone from GitHub)
# ====================================================================
import shutil
import os

# Option A: Copy from Drive (if you uploaded the project folder)
if os.path.exists(DRIVE_PROJECT_DIR):
    if os.path.exists(LOCAL_PROJECT_DIR):
        shutil.rmtree(LOCAL_PROJECT_DIR)
    shutil.copytree(DRIVE_PROJECT_DIR, LOCAL_PROJECT_DIR)
    print("Copied project from Drive.")
else:
    # Option B: Clone from GitHub (update URL)
    !git clone https://github.com/YOUR_USERNAME/DepressoSpeech.git {LOCAL_PROJECT_DIR}

%cd {LOCAL_PROJECT_DIR}/Model
!ls -la

In [ ]:
# ====================================================================
# CELL 4: Organize Patient Data (if not already organized)
# ====================================================================
# Set the parent folder containing audio/, transcripts/, features/
SOURCE_DIR = "/content/drive/MyDrive/DepressoSpeech/Model"  # UPDATE THIS
OUTPUT_DIR = "data/patients"

import sys
sys.path.insert(0, "/content/DepressoSpeech/Model")

from scripts.organize_patient_data import organize_patient_data, generate_patient_manifest
from pathlib import Path

if not Path(OUTPUT_DIR).exists() or not any(Path(OUTPUT_DIR).iterdir()):
    registry = organize_patient_data(
        source_dir=Path(SOURCE_DIR),
        output_dir=Path(OUTPUT_DIR),
        copy_mode=False,  # Use symlinks (faster on Colab)
    )
    manifest = generate_patient_manifest(Path(OUTPUT_DIR))
    print(f"Organized {len(manifest)} patients.")
    display(manifest.head())
else:
    print("Patient data already organized.")
    manifest = generate_patient_manifest(Path(OUTPUT_DIR))
    print(f"Found {len(manifest)} patients.")

In [ ]:
# ====================================================================
# CELL 5: Verify GPU and Estimate Training Time
# ====================================================================
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU not available! Change runtime to GPU: Runtime > Change runtime type > GPU")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"\nGPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

# Rough estimates per epoch
if 'T4' in gpu_name:
    sec_per_epoch = 3
elif 'V100' in gpu_name:
    sec_per_epoch = 1.5
elif 'A100' in gpu_name:
    sec_per_epoch = 1
else:
    sec_per_epoch = 5

max_epochs = 500 + 300 + 200  # stage1 + stage2 + stage3 max
est_minutes = (max_epochs * sec_per_epoch) / 60
print(f"Estimated max training time: ~{est_minutes:.0f} minutes (with early stopping, usually much less)")

In [ ]:
# ====================================================================
# CELL 6: Run Trimodal Training
# ====================================================================
import subprocess
import sys

# You can override config values here
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'

cmd = [
    sys.executable,
    "scripts/train_trimodal.py",
    "--config", "configs/trimodal_v2_config.yaml",
    "--patients-dir", "data/patients",
    "--splits-dir", "data/splits",
]

print("Starting training...")
print("This may take 30-90 minutes depending on GPU and early stopping.\n")

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

process.wait()
if process.returncode != 0:
    print(f"\nTraining exited with code {process.returncode}")

In [ ]:
# ====================================================================
# CELL 7: Save Checkpoints to Google Drive
# ====================================================================
import shutil
from pathlib import Path
from datetime import datetime

DRIVE_CHECKPOINT_DIR = Path(DRIVE_PROJECT_DIR) / "checkpoints" / datetime.now().strftime("%Y%m%d_%H%M%S")
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

local_ckpt_dir = Path("checkpoints")

for ckpt_file in local_ckpt_dir.glob("*.pt"):
    dest = DRIVE_CHECKPOINT_DIR / ckpt_file.name
    shutil.copy2(str(ckpt_file), str(dest))
    print(f"Saved: {ckpt_file.name} -> {dest}")

# Also save config and logs
for log_file in Path("logs").glob("*.log"):
    dest = DRIVE_CHECKPOINT_DIR / log_file.name
    shutil.copy2(str(log_file), str(dest))

print(f"\nAll checkpoints saved to: {DRIVE_CHECKPOINT_DIR}")

In [ ]:
# ====================================================================
# CELL 8: Quick Inference Test (optional)
# ====================================================================
from src.inference.trimodal_pipeline import TrimodalPredictor
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
predictor = TrimodalPredictor(
    checkpoint_path="checkpoints/best_trimodal.pt",
    device=str(device),
)

# Test on first patient
patient_dirs = sorted([d for d in Path("data/patients").iterdir() if d.is_dir()])
if patient_dirs:
    test_patient = patient_dirs[0]
    result = predictor.predict_from_patient_folder(test_patient)
    print(f"Patient: {result.participant_id}")
    print(f"  PHQ-8 Score: {result.phq8_score}")
    print(f"  Severity: {result.severity}")
    print(f"  Modalities: {result.modalities_used}")
    print(f"  Contributions: {result.modality_contributions}")
else:
    print("No patients found.")